In [ ]:
# # The maths behind XGBoost: numerical checks
# We check the derivation of the Note with numbers: the second-order Taylor approximation, the optimal leaf output
# w* = -G / (H + lambda), the minimum objective, the gain formula, and how the XGBoost library scales its gain.

In [ ]:
import numpy as np
import xgboost as xgb
sigmoid = lambda z: 1 / (1 + np.exp(-z))

## 1. Squared error: the second-order Taylor approximation is exact

In [ ]:
# one leaf of the regression Note: two students with residuals -2.875 and -1.375 from the mean 7.375
y = np.array([4.5, 6.0])
y_prev = np.full(2, 7.375)
loss = lambda w: (0.5 * (y - (y_prev + w)) ** 2).sum()     # L = 1/2 (y - yhat)^2
g = y_prev - y                                             # gradient: yhat - y
h = np.ones(2)                                             # Hessian: 1
taylor = lambda w: loss(0) + g.sum() * w + 0.5 * h.sum() * w ** 2
ws = np.linspace(-4, 2, 13)
print("largest gap between exact and Taylor:", max(abs(loss(w) - taylor(w)) for w in ws))

## 2. w* = -G / (H + lambda), checked by brute force on a fine grid

In [ ]:
grid = np.linspace(-5, 5, 200001)
for lam in (0, 1):
    G, H = g.sum(), h.sum()
    w_star = -G / (H + lam)
    objective = [loss(w) + 0.5 * lam * w ** 2 for w in grid]
    print(f"lambda={lam}: formula w*={w_star:.4f}, grid minimum at {grid[np.argmin(objective)]:.4f}")

## 3. Log loss: the approximation is close, not exact

In [ ]:
# left leaf of the classification Note: classes 0, 1, 0, all at log-odds ln 1.5
yc = np.array([0, 1, 0])
z0 = np.log(1.5)
p = sigmoid(z0)
logloss = lambda w: -(yc * np.log(sigmoid(z0 + w)) + (1 - yc) * np.log(1 - sigmoid(z0 + w))).sum()
G, H = (p - yc).sum(), len(yc) * p * (1 - p)             # g = p - y, h = p(1 - p) for each of the 3 rows
approx = lambda w: logloss(0) + G * w + 0.5 * H * w ** 2
for w in (-2, -1.11, 0, 1):
    print(f"w={w:+.2f}: exact {logloss(w):.4f}, second-order {approx(w):.4f}")
w_star = -G / H
w_grid = grid[np.argmin([logloss(w) for w in grid])]
print(f"w* from the formula {w_star:.4f}; true minimum of the log loss {w_grid:.4f}")

## 4. Minimum objective and gain on the regression data

In [ ]:
x = np.array([6.7, 9.0, 7.5, 5.0])
yr = np.array([4.5, 11.0, 6.0, 8.0])
gr = np.full(4, yr.mean()) - yr                            # gradients
hr = np.ones(4)                                            # Hessians

def best_obj(mask, lam=0.0, gamma=0.0):
    # -1/2 G^2 / (H + lambda) + gamma * T, for one leaf (T = 1)
    G, H = gr[mask].sum(), hr[mask].sum()
    return -0.5 * G ** 2 / (H + lam) + gamma

def gain(mask, parent, lam=0.0, gamma=0.0):
    # 1/2 [G_L^2/(H_L+lam) + G_R^2/(H_R+lam) - G_P^2/(H_P+lam)] - gamma
    sim = lambda m: gr[m].sum() ** 2 / (hr[m].sum() + lam)
    return 0.5 * (sim(mask & parent) + sim(~mask & parent) - sim(parent)) - gamma

everyone = np.ones(4, bool)
left = x < 8.25
print("objective before the split:", best_obj(everyone))
print("objective after the split :", best_obj(left) + best_obj(~left))
print("paper gain (with 1/2)     :", gain(left, everyone))
print("similarity gain (no 1/2)  :", 2 * gain(left, everyone))
print("lower split, paper gain   :", gain(x < 5.85, left))

## 5. What the XGBoost library prints, and how it compares gamma
The dumped gain is the similarity gain without the 1/2. Gamma is compared with that number:
the lower split (gain 5.04) survives gamma = 5.0 and disappears at gamma = 5.1.

In [ ]:
X = x.reshape(-1, 1)
for gamma in (0, 5.0, 5.1, 10, 18):
    m = xgb.XGBRegressor(n_estimators=1, max_depth=2, learning_rate=1.0, reg_lambda=0, gamma=gamma,
                         min_child_weight=0, base_score=yr.mean(), tree_method="exact", device="cpu").fit(X, yr)
    print(f"gamma={gamma}:")
    print(m.get_booster().get_dump(with_stats=True)[0])